# 레인 C-0 · 설치줄 점검

**무엇을 하는가** — 레인 B(월간 자동 실행)는 노트북의 `!pip` 줄을 **지우고** 최신판으로 돌린다.
그래서 **설치 줄이 무엇이든 초록불이 뜬다.** 실제로 `sentencepiece`와 `accelerate`가
빠져 있었는데도 몇 달 동안 초록불이었다. 이 노트북이 그 구멍을 메운다.

**어디서 도는가** — Colab. **GPU 불필요**(런타임 유형은 CPU 그대로 두면 된다).

**어떻게 쓰는가** — 위에서부터 셀을 차례로 실행한다.
중간에 **런타임을 다시 시작하라는 안내가 떠도 무시하고 다음 셀로 간다.**
설치한 것을 이 노트북 자신이 import 하지 않고 **별도 파이썬을 띄워 시험하기 때문에**
다시 시작할 필요가 없다.

**마지막 셀이 표 두 개를 찍는다.** 위 표는 코드가 판정한 것, 아래 표는 저자가 판단할 것이다.
**두 표를 통째로 복사해 지침서에 붙여넣는다.**

---

In [ ]:
# ── 셀 1 · 설치 전 Colab 기본 환경 실측 ────────────────────────────────────────
# 이 값이 "독자가 아무것도 설치하지 않은 상태"다. 개정판 판번호 조사의 '설치 전' 값이기도 하다.

import importlib.metadata as md, importlib.util, sys, shutil, platform

BEFORE = {}

def dist_ver(name):
    try:
        return md.version(name)
    except Exception:
        return None

# 왼쪽=배포판 이름(pip 이름) / 오른쪽=import 이름
WATCH = [
    ("transformers", "transformers"), ("gradio", "gradio"),
    ("diffusers", "diffusers"), ("accelerate", "accelerate"),
    ("sentence-transformers", "sentence_transformers"),
    ("sentencepiece", "sentencepiece"), ("bertviz", "bertviz"),
    ("gTTS", "gtts"), ("librosa", "librosa"), ("ffmpeg-python", "ffmpeg"),
    ("torch", "torch"), ("numpy", "numpy"), ("pandas", "pandas"),
    ("tensorflow", "tensorflow"), ("keras", "keras"),
    ("matplotlib", "matplotlib"), ("scikit-learn", "sklearn"), ("seaborn", "seaborn"),
    ("gymnasium", "gymnasium"), ("gspread", "gspread"),
    ("ipywidgets", "ipywidgets"), ("ipython", "IPython"),
    ("tensorflow-datasets", "tensorflow_datasets"),
    ("huggingface-hub", "huggingface_hub"), ("click", "click"),
    ("soundfile", "soundfile"), ("pillow", "PIL"),
]

# 사각지대 — 설치 줄이 없어서 판번호 고정 정책이 구조적으로 볼 수 없는 것들
BLIND = {"gymnasium", "gspread", "ipywidgets", "tensorflow-datasets"}

print(f"Python {platform.python_version()}   /   pip {dist_ver('pip')}")
print(f"ffmpeg 바이너리: {shutil.which('ffmpeg') or '(없음)'}")
print()
# 노트북이 실제로 부르는 대표 함수. **있는 것과 쓸 수 있는 것은 다르다.**
# tensorflow_datasets 4.9.10 의 __init__.py 는 내부 import 실패를 try/except 로
# 삼킨다 → `import` 는 성공하는데 모듈이 텅 빈 채로 남는다.
# find_spec 만 보면 ○ 가 뜨고, 독자는 `tfds.load` 에서 AttributeError 를 만난다.
# (2026-08-21 레인 C-1 에서 VI-4·VI-5 가 정확히 이렇게 죽었다)
PROBE = {
    "tensorflow_datasets": "load", "gymnasium": "make", "gspread": "authorize",
    "ipywidgets": "Button", "transformers": "pipeline", "gradio": "Interface",
    "torch": "tensor", "tensorflow": "constant", "numpy": "array",
    "sklearn": None, "librosa": "load", "gtts": "gTTS",
    "sentence_transformers": "SentenceTransformer", "diffusers": "StableDiffusionPipeline",
    "accelerate": "Accelerator", "huggingface_hub": "hf_hub_download",
}

def probe(imp_name):
    """정말 쓸 수 있는지 본다. (표시, 문제 설명) 반환."""
    if importlib.util.find_spec(imp_name) is None:
        return "×", "모듈 없음"
    try:
        mod = importlib.import_module(imp_name)
    except Exception as e:
        return "🔴", f"import 실패: {type(e).__name__}: {e}"
    attr = PROBE.get(imp_name, "__name__")
    if attr and not hasattr(mod, attr):
        n = len([x for x in dir(mod) if not x.startswith("_")])
        return "🔴", f"import 은 됐는데 .{attr} 가 없다 (공개 항목 {n}개) — 속이 빈 모듈이다"
    return "○", ""

print(f"{'패키지':<24}{'판번호':<16}{'쓸 수 있나':<12}")
print("-" * 60)
BROKEN = []
for pip_name, imp_name in WATCH:
    v = dist_ver(pip_name)
    BEFORE[pip_name] = v
    ok, why = probe(imp_name)
    mark = "  ← 사각지대" if pip_name in BLIND else ""
    print(f"{pip_name:<24}{(v or '(없음)'):<16}{ok:<12}{mark}")
    if why and ok == "🔴":
        print(f"{'':<24}🔴 {why}")
        BROKEN.append((pip_name, why))

print()
print("※ '(없음)'인데 ○ 이면 Colab 이 다른 이름으로 넣어 둔 것이다.")
print("※ 🔴 는 **깔려 있는데 못 쓰는** 것이다. 판번호만 보면 절대 안 보인다.")
print("※ 사각지대 네 개가 '(없음)'이나 🔴 이면 그 노트북은 첫 줄에서 막힌다.")
if BROKEN:
    print()
    print(f"🔴 못 쓰는 패키지 {len(BROKEN)}개 — 이 노트북들은 지금 독자가 실행할 수 없다:")
    for n, w in BROKEN:
        print(f"   {n}: {w}")

In [ ]:
# ── 셀 2 · 저장소를 받아 셸 줄을 스스로 센다 ──────────────────────────────────
# 개수를 손으로 적지 않는다. 지침서의 "활성 pip 줄 14개"가 지금도 14개인지 여기서 확인된다.

import subprocess, tarfile, io, json, glob, os, urllib.request, ssl

TGZ = "https://codeload.github.com/MLFundamentals/making-ai/tar.gz/refs/heads/main"
subprocess.run(["curl", "-sL", "-o", "/content/repo.tgz", TGZ], check=True)
with tarfile.open("/content/repo.tgz") as t:
    t.extractall("/content/repo")
ROOT = glob.glob("/content/repo/making-ai-*")[0]
NB_DIR = os.path.join(ROOT, "notebooks")

NOTEBOOKS = sorted(glob.glob(os.path.join(NB_DIR, "*.ipynb"))) + \
            sorted(glob.glob(os.path.join(NB_DIR, "python-basics", "*.ipynb")))

def rel(p):
    return os.path.relpath(p, NB_DIR)

SHELL = []          # (노트북, 셀번호, 줄)
PIP_OF = {}         # 노트북 → [pip 줄]
for p in NOTEBOOKS:
    nb = json.load(open(p, encoding="utf-8"))
    pips = []
    ci = -1
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        ci += 1
        for ln in "".join(c["source"]).splitlines():
            s = ln.strip()
            if s.startswith("!") or s.startswith("%pip"):
                SHELL.append((rel(p), ci, s))
                if "pip" in s:
                    pips.append(s)
    PIP_OF[rel(p)] = pips

PIP_LINES = [s for _, _, s in SHELL if "pip" in s]
OTHER_SHELL = [(f, i, s) for f, i, s in SHELL if "pip" not in s]

def norm(line):
    """'!pip -q install a==1 b==2' → 'a==1 b==2' (순서 무시 비교용)"""
    toks = line.lstrip("!%").split()
    pkgs = [t for t in toks if "==" in t or (t and not t.startswith("-") and t not in ("pip", "install"))]
    return " ".join(sorted(pkgs))

DISTINCT = {}
for line in PIP_LINES:
    DISTINCT.setdefault(norm(line), line)

print(f"노트북      {len(NOTEBOOKS)}편  (최상위 {len(glob.glob(os.path.join(NB_DIR,'*.ipynb')))} + python-basics {len(glob.glob(os.path.join(NB_DIR,'python-basics','*.ipynb')))})")
print(f"셸 줄       {len(SHELL)}줄")
print(f"  pip 줄    {len(PIP_LINES)}줄  → 중복 제거 {len(DISTINCT)}종")
print(f"  그 밖      {len(OTHER_SHELL)}줄")
print()
print("── 설치 줄 " + str(len(DISTINCT)) + "종 ──")
for i, (k, v) in enumerate(DISTINCT.items(), 1):
    users = [f for f, ps in PIP_OF.items() if any(norm(p) == k for p in ps)]
    print(f"{i:>2}. {v}")
    print(f"    쓰는 노트북: {', '.join(users)}")
print()
print("── pip 이 아닌 셸 줄 ──")
for f, i, s in OTHER_SHELL:
    print(f"    {f} [셀 {i}]  {s}")

print()
print("※ 위 개수가 지침서와 다르면 노트북이 바뀐 것이다 — 지침서를 고쳐야 한다.")

In [ ]:
# ── 셀 3 · 고정 판번호가 실제로 설치 가능한가 (설치는 하지 않는다) ───────────
# --dry-run 은 해결만 하고 내려받지 않는다. 줄끼리 서로 오염시키지 않으므로
# 어느 한 줄이 깨져도 나머지를 전부 볼 수 있다. PyPI 에서 판번호가 내려간 경우를 잡는다.

import subprocess, sys, tempfile, json, os, time

DRYRUN = {}
for i, (key, line) in enumerate(DISTINCT.items(), 1):
    pkgs = [t for t in line.lstrip("!%").split()
            if t not in ("pip", "install") and not t.startswith("-")]
    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tf:
        rp = tf.name
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--dry-run", "-q",
         "--report", rp, *pkgs],
        capture_output=True, text=True)
    dt = time.time() - t0
    resolved = []
    if r.returncode == 0:
        try:
            rep = json.load(open(rp))
            for it in rep.get("install", []):
                m = it.get("metadata", {})
                resolved.append(f"{m.get('name')}=={m.get('version')}")
        except Exception:
            pass
    os.unlink(rp)
    DRYRUN[key] = dict(ok=(r.returncode == 0), err=r.stderr.strip()[-600:],
                       n_new=len(resolved), secs=round(dt, 1))
    flag = "OK  " if r.returncode == 0 else "실패"
    print(f"{flag} [{i:>2}/{len(DISTINCT)}] {line}")
    print(f"        새로 받을 패키지 {len(resolved)}개 · 해결에 {dt:.1f}초")
    if r.returncode != 0:
        print("        " + r.stderr.strip().replace("\n", "\n        ")[-600:])

print()
bad = [k for k, v in DRYRUN.items() if not v["ok"]]
print("설치 불가한 줄:", (str(len(bad)) + "개 — 위 '실패' 참조") if bad else "없음")

In [ ]:
# ── 셀 4 · pip 아닌 셸 줄의 대상이 살아 있는가 ────────────────────────────────
# GloVe 는 http 이고 레인 A 의 컨테이너에서는 확인할 수 없다. Colab 은 독자와 같은 자리다.

import urllib.request, re

URLS = []
for f, i, s in OTHER_SHELL:
    for m in re.findall(r"https?://\S+", s):
        URLS.append((f, m.rstrip("'\"")))

SHELL_URL = {}
for f, u in URLS:
    try:
        req = urllib.request.Request(u, method="HEAD",
                                     headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            code, final = resp.status, resp.url
            size = resp.headers.get("Content-Length")
    except Exception as e:
        code, final, size = f"에러 {type(e).__name__}", "", None
    mb = f"{int(size)/1024/1024:.0f}MB" if size and str(size).isdigit() else "?"
    SHELL_URL[u] = dict(code=code, final=final, mb=mb, nb=f)
    print(f"{f}\n    {u}\n    → {code}  {mb}  {('리다이렉트 ' + final) if final and final != u else ''}")

if not URLS:
    print("URL 이 있는 셸 줄이 없다.")

In [ ]:
# ── 셀 5 · 설치 줄에 없는데 import 하는 것이 있는가 (설치 전에 잰다) ────────
# 이 셀은 "import 하는데 안 깔린 것"을 잡는다.
# ⚠ 반대쪽 구멍은 못 잡는다 — accelerate 나 sentencepiece 처럼
#   노트북이 직접 import 하지 않고 라이브러리가 속으로 부르는 것은 여기 안 걸린다.
#   그건 C-1(실제 실행)의 일이다.

import ast, sys, importlib.util

ALIAS = {"cv2": "opencv-python", "PIL": "pillow", "sklearn": "scikit-learn",
         "tensorflow_datasets": "tensorflow-datasets", "gtts": "gTTS",
         "ffmpeg": "ffmpeg-python", "sentence_transformers": "sentence-transformers",
         "huggingface_hub": "huggingface-hub", "google": "google-colab",
         "mpl_toolkits": "matplotlib", "IPython": "ipython", "yaml": "pyyaml",
         "skimage": "scikit-image"}
STD = set(sys.stdlib_module_names)

def imports_of(path):
    nb = json.load(open(path, encoding="utf-8"))
    code = []
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        for ln in "".join(c["source"]).splitlines():
            code.append("pass" if ln.strip().startswith(("!", "%")) else ln)
    try:
        tree = ast.parse("\n".join(code))
    except SyntaxError:
        return None
    mods = set()
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            for a in n.names:
                mods.add(a.name.split(".")[0])
        elif isinstance(n, ast.ImportFrom) and n.level == 0 and n.module:
            mods.add(n.module.split(".")[0])
    return sorted(m for m in mods if m not in STD)

MISSING = {}   # 노트북 → [설치 줄에도 없고 Colab 기본에도 없는 모듈]
for p in NOTEBOOKS:
    f = rel(p)
    mods = imports_of(p)
    if mods is None:
        print(f"⚠ 파싱 실패: {f}")
        continue
    joined = " ".join(PIP_OF[f]).lower()
    gap = []
    for m in mods:
        in_pip = ALIAS.get(m, m).lower() in joined or m.lower() in joined
        in_colab = importlib.util.find_spec(m) is not None
        if not in_pip and not in_colab:
            gap.append(m)
    if gap:
        MISSING[f] = gap

if MISSING:
    print("🔴 설치 줄에도 없고 Colab 기본에도 없다 — 독자가 첫 줄에서 막힌다")
    for f, g in MISSING.items():
        print(f"    {f}\n        {g}")
else:
    print("✅ 모든 노트북의 import 가 '설치 줄' 또는 'Colab 기본'으로 덮인다")

print()
NO_PIP = [f for f in PIP_OF if not PIP_OF[f]]
print(f"참고 · 설치 줄이 아예 없는 노트북 {len(NO_PIP)}편 — 전부 Colab 기본에 기대고 있다.")
print("      이 편들은 판번호 고정 정책이 구조적으로 볼 수 없는 자리다(지침서 3장 '사각지대').")

In [ ]:
# ── 셀 6 · 실제로 설치하고 별도 파이썬으로 import 해 본다 ────────────────────
# gTTS 줄을 맨 뒤로 돌린다. gTTS 가 click 을 내려 huggingface-hub 를 망가뜨리는지
# 확인하려면 다른 줄이 먼저 깔려 있어야 한다.
# import 시험은 subprocess 로 한다 → 런타임을 다시 시작할 필요가 없다.

import subprocess, sys, time, json

order = sorted(DISTINCT.items(), key=lambda kv: ("gtts" in kv[0].lower()))

IMP_NAME = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",
            "gTTS": "gtts", "ffmpeg-python": "ffmpeg", "tensorflow-datasets":
            "tensorflow_datasets", "huggingface-hub": "huggingface_hub",
            "pillow": "PIL", "ipython": "IPython"}

def try_import(mods):
    code = "\n".join(f"import {m}" for m in mods) + "\nprint('OK')"
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return r.returncode == 0, (r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "")

HF_PROBE = ("import importlib.metadata as m;import huggingface_hub;"
            "print('click', m.version('click'));"
            "print('huggingface_hub', huggingface_hub.__version__)")

def hf_state():
    r = subprocess.run([sys.executable, "-c", HF_PROBE], capture_output=True, text=True)
    return (r.returncode == 0), (r.stdout.strip() or r.stderr.strip().splitlines()[-1])

INSTALL = {}
hf_before_gtts = None       # gTTS 줄 '직전'의 huggingface-hub 상태
for i, (key, line) in enumerate(order, 1):
    is_gtts = "gtts" in key.lower()
    if is_gtts:
        hf_before_gtts = hf_state()
        print(f"      (gTTS 줄 직전 huggingface-hub: "
              f"{'정상' if hf_before_gtts[0] else '이미 못 씀'} — {hf_before_gtts[1]})")

    pkgs = [t for t in line.lstrip("!%").split()
            if t not in ("pip", "install") and not t.startswith("-")]
    names = [p.split("==")[0] for p in pkgs]
    mods = [IMP_NAME.get(n, n.replace("-", "_")) for n in names]

    print(f"[{i}/{len(order)}] {line}")
    t0 = time.time()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    dt = time.time() - t0
    ok_imp, imp_err = try_import(mods)
    raw = (r.stderr.strip() or r.stdout.strip())
    # 앞에서 자르면 resolver 충돌이, 뒤에서 자르면 'error:' 첫 줄이 사라진다 → 양쪽을 남긴다
    ins_err = raw if len(raw) <= 1200 else raw[:500] + "\n      … (중략) …\n" + raw[-600:]
    INSTALL[key] = dict(line=line, install_ok=(r.returncode == 0), import_ok=ok_imp,
                        secs=round(dt, 1), err=(ins_err if r.returncode else imp_err))
    print(f"      설치 {'OK' if r.returncode==0 else '실패'} ({dt:.0f}초) · "
          f"import {'OK' if ok_imp else '실패'}")
    # 실패 사유를 절대 삼키지 않는다 — 조용한 실패가 이 프로젝트의 유일한 사고 유형이다
    if r.returncode != 0:
        print("      ↳ 설치 오류: " + ins_err.replace("\n", "\n        "))
    if not ok_imp:
        print("      ↳ import 오류: " + imp_err)

# gTTS 오염 확인 — '깨졌다'가 아니라 '멀쩡하던 것이 깨졌다'를 본다.
# 전 상태를 안 재고 후 상태만 보면, 원래 없던 것도 '오염'으로 잘못 읽힌다.
print()
print("── gTTS 오염 확인 ──")
hf_after = hf_state()
if hf_before_gtts is None:
    GTTS_CONTAM = None
    print("판정: 측정 못 함 (gTTS 설치 줄이 목록에 없다)")
elif not hf_before_gtts[0]:
    GTTS_CONTAM = None
    print(f"판정: 측정 못 함 — gTTS 를 깔기 전에 이미 huggingface-hub 를 못 쓰는 상태였다")
    print(f"      전: {hf_before_gtts[1]}")
    print(f"      후: {hf_after[1]}")
else:
    GTTS_CONTAM = not hf_after[0]
    print(f"      전: {hf_before_gtts[1]}")
    print(f"      후: {hf_after[1]}")
    print("판정:", "🔴 멀쩡하던 huggingface-hub 가 gTTS 때문에 깨졌다"
          if GTTS_CONTAM else "✅ 한 pip 명령에 넣으면 깨지지 않는다 (경고만 시끄럽다)")

In [ ]:
# ── 셀 7 · 판정표 두 칸 ───────────────────────────────────────────────────────
# 아래 출력 전체를 복사해 지침서에 붙여넣는다.

import importlib.metadata as md, datetime, platform

def dv(n):
    try:
        return md.version(n)
    except Exception:
        return "(없음)"

print("=" * 74)
print(f"레인 C-0 · 설치줄 점검   {datetime.date.today()}   Python {platform.python_version()}")
print("=" * 74)
print()
print("[코드가 판정한 것]")
print()
print(f"  대상            노트북 {len(NOTEBOOKS)}편 · 셸 줄 {len(SHELL)}줄 · pip 줄 {len(PIP_LINES)}줄 ({len(DISTINCT)}종)")
dry_bad = [k for k, v in DRYRUN.items() if not v['ok']]
print(f"  판번호 해결      {len(DISTINCT)-len(dry_bad)}/{len(DISTINCT)} 종 OK" + (f"  🔴 실패 {len(dry_bad)}종" if dry_bad else ""))
ins_bad = [k for k, v in INSTALL.items() if not (v['install_ok'] and v['import_ok'])]
print(f"  설치+import     {len(INSTALL)-len(ins_bad)}/{len(INSTALL)} 종 OK" + (f"  🔴 실패 {len(ins_bad)}종" if ins_bad else ""))
print(f"  누락 import     {'없음' if not MISSING else '🔴 ' + str(len(MISSING)) + '편'}")
print(f"  gTTS 오염       {'측정 못 함 ⚠' if GTTS_CONTAM is None else ('🔴 huggingface-hub 깨짐' if GTTS_CONTAM else '없음 (경고만)')}")
print()
print("  ── 설치 시간 (독자가 첫 셀에서 기다리는 시간) ──")
for k, v in sorted(INSTALL.items(), key=lambda kv: -kv[1]['secs']):
    print(f"    {v['secs']:>6.0f}초  {v['line']}")
print()
print("  ── 사각지대 네 개의 Colab 기본 판번호 ──")
for n in ["gymnasium", "gspread", "ipywidgets", "tensorflow-datasets"]:
    print(f"    {n:<22}{BEFORE.get(n) or '🔴 (없음)'}")
print()
print("  ── 설치 후 판번호 (설치로 바뀐 것만 · 개정판 조사용) ──")
_changed = 0
for pip_name, _ in WATCH:
    b = BEFORE.get(pip_name) or "(없음)"      # None 과 '(없음)' 을 같은 값으로 맞춘다
    a = dv(pip_name)
    if b != a:
        print(f"    {pip_name:<22}{b:<14}→ {a}")
        _changed += 1
if _changed == 0:
    print("    바뀐 것 없음 — 고정값이 이미 Colab 기본과 같다")
print()
if MISSING:
    print("  ── 🔴 누락 상세 ──")
    for f, g in MISSING.items():
        print(f"    {f}: {g}")
    print()
print("-" * 74)
print()
print("[저자가 판단할 것 — 답을 적고 지침서에 함께 붙여넣는다]")
print()
print("  □ 사각지대 중 '(없음)'이 나온 것이 있는가?")
print("      → 있다면 그 노트북에 설치 줄을 넣어야 한다.")
print("        비용은 원고다 — 코드 목록 줄번호가 전부 밀리고 본문 참조도 따라 움직인다.")
print("      답: ")
print()
print("  □ 설치 시간이 가장 긴 줄이 몇 초였는가? 독자에게 안내할 만한 길이인가?")
print("      → 1분을 넘으면 원고에 '설치에 약 N분 걸립니다' 한 줄이 필요하다.")
print("      답: ")
print()
print("  □ VII-1(Generative AI_Text Generator)에 transformers 설치 줄을 넣을 것인가?")
print("      → 지침서 3장의 미결 항목. 이 노트북의 측정값이 근거가 된다.")
print("      답: ")
print()
print("  □ tensorflow-datasets 를 판번호 고정 대상에 넣을 것인가?")
print("      → cats_vs_dogs·imdb_reviews 의 다운로드 주소가 이 패키지 안에 박혀 있다.")
print("        판번호가 바뀌면 받는 파일이 바뀐다.")
print("      답: ")
print()
print("  □ 위 '설치 후 판번호' 표에서 고정값과 다른 값이 깔린 것이 있는가?")
print("      → 있다면 고정이 듣지 않은 것이다. 즉시 조사해야 한다.")
print("      답: ")
print("=" * 74)